In [1]:
import scarf

scarf.set_verbosity('WARNING')

In [2]:
datasets = scarf.cytebase.connect("scarf_docs")
datasets.list_datasets()

['annotations',
 'baron_8K_pancreas_rnaseq',
 'bastidas-ponce_4K_pancreas-d15_rnaseq',
 'cao_2.1M_moca_rnaseq',
 'cusanovich_81K_mouse_atacseq',
 'hca_783K_blood_rnaseq',
 'kang_14K_ifnb-pbmc_rnaseq',
 'kang_15K_pbmc_rnaseq',
 'lecun_60K_mnist_images',
 'motifs',
 'muraro_2K_pancreas_rnaseq',
 'saunders_110K_brain_rnaseq',
 'segerstolpe_2K_pancreas_rnaseq',
 'tenx_1.3M_brain_rnaseq',
 'tenx_10K_pbmc-v1_atacseq',
 'tenx_3K_pbmc_multiome-gex-atac',
 'tenx_5K_pbmc_rnaseq',
 'tenx_8K_pbmc_citeseq',
 'xin_1K_pancreas_rnaseq',
 'zalando_60K_fmnist_images',
 'zeisel_161K_nervous_rnaseq',
 'zheng_69K_pbmc_rnaseq']

In [3]:
# This dataset is in Cellranger (10x) HDF5 format.
datasets.download_dataset(
    name='tenx_10K_pbmc-v1_atacseq',
    destination='./scarf_datasets'
)

PosixPath('/tmp/tmpn0kyp8c6/scarf_datasets/tenx_10K_pbmc-v1_atacseq')

In [4]:
# This dataset is in MTX format along with barcodes and features TSV files.
datasets.download_dataset(
    name='xin_1K_pancreas_rnaseq',
    destination='./scarf_datasets'
)

PosixPath('/tmp/tmpn0kyp8c6/scarf_datasets/xin_1K_pancreas_rnaseq')

In [5]:
# This dataset is in H5ad (anndata) format.
datasets.download_dataset(
    name='bastidas-ponce_4K_pancreas-d15_rnaseq',
    destination='./scarf_datasets'
)

PosixPath('/tmp/tmpn0kyp8c6/scarf_datasets/bastidas-ponce_4K_pancreas-d15_rnaseq')

In [6]:
# Assay type is inferred from the H5 contents (RNA, ATAC, or multimodal).
reader = scarf.CrH5Reader(
    'scarf_datasets/tenx_10K_pbmc-v1_atacseq/data.h5'
)

# change value of `zarr_loc` to your choice of filename and path
writer = scarf.CrToZarr(
    reader,
    zarr_loc='scarf_datasets/pbmc_atac.zarr'  
)  
writer.dump()

In [7]:
 # Note here we only give name of directory containing MTX file (along with barcodes and features file)
reader = scarf.CrDirReader(
    'scarf_datasets/xin_1K_pancreas_rnaseq'
)

# change value of `zarr_loc` to your choice of filename and path
writer = scarf.CrToZarr(
    reader, 
    zarr_loc='scarf_datasets/xin_1K.zarr'
)
writer.dump()

feature_types extraction failed from features.tsv.gz in column 2



In [8]:
# H5adReader takes the path to the .h5ad file directly.
# In this catalog file, the feature index contains the gene names.
reader = scarf.H5adReader(
    'scarf_datasets/bastidas-ponce_4K_pancreas-d15_rnaseq/data.h5ad',
    cell_ids_key='index',
    feature_ids_key='index',
    feature_name_key='index',
)

# change value of `zarr_loc` to your choice of filename and path
writer = scarf.H5adToZarr(
    reader,
    zarr_loc='scarf_datasets/differentiating_pancreatic_cells.zarr'
)
writer.dump()

In [9]:
ds = scarf.DataStore('scarf_datasets/differentiating_pancreatic_cells.zarr')

In [10]:
scarf.writers.to_mtx(
    assay=ds.RNA,
    mtx_directory='scarf_datasets/diff_pancreas'
)

In [11]:
ds = scarf.DataStore('scarf_datasets/differentiating_pancreatic_cells.zarr')

In [12]:
scarf.writers.to_h5ad(
    assay=ds.RNA,
    h5ad_filename='scarf_datasets/diff_pancreas.h5ad'
)

In [13]:
from pathlib import Path
import shutil

import numpy as np
from scipy.sparse import csr_matrix

csv_dir = Path('scarf_datasets')
csv_dir.mkdir(parents=True, exist_ok=True)
csv_path = csv_dir / 'toy_counts.csv'
csv_path.write_text(
    'quality,geneA,geneB,geneC\n'
    '10,1,0,2\n'
    '20,0,3,0\n'
    '30,4,5,6\n'
    '40,7,0,8\n'
    '50,9,10,0\n',
    encoding='utf-8',
)

csv_zarr = csv_dir / 'toy_csv.zarr'
if csv_zarr.exists():
    shutil.rmtree(csv_zarr)

reader = scarf.CSVReader(
    str(csv_path),
    cell_data_cols=['quality'],
)
writer = scarf.CSVtoZarr(
    reader,
    zarr_loc=str(csv_zarr),
    assay_name='RNA',
    dtype=np.dtype('uint16'),
)
writer.dump()
ds_csv = scarf.DataStore(str(csv_zarr))
ds_csv

Minimum cell count (3) is lower than size factor multiplier (1000)



No matches found for pattern MT-|mt. Will not add/update percentage feature



No matches found for pattern RPS|RPL|MRPS|MRPL. Will not add/update percentage feature



More than of half of the less have less than 10 features for assay: RNA. Will not remove low quality cells automatically.



DataStore has 5 (5) cells with 1 assays: RNA
   Cell metadata:
            'I', 'ids', 'names', 'RNA_nFeatures', 'quality', 
            'RNA_nCounts'
   RNA assay has 0 (3) features and following metadata:
            'I', 'ids', 'names', 'nCells', 'dropOuts', 
          

In [14]:
mat = csr_matrix(
    (
        [1, 10, 15, 10, 20, 2, 3, 1, 5],
        ([0, 0, 0, 1, 1, 1, 2, 2, 2], [1, 3, 8, 2, 3, 1, 2, 8, 9]),
    ),
    shape=(3, 10),
)
sparse_zarr = Path('scarf_datasets/toy_sparse.zarr')
if sparse_zarr.exists():
    shutil.rmtree(sparse_zarr)
sparse_writer = scarf.SparseToZarr(
    mat,
    zarr_loc=str(sparse_zarr),
    cell_ids=[f'cell_{i}' for i in range(mat.shape[0])],
    feature_ids=[f'feat_{i}' for i in range(mat.shape[1])],
    assay_name='RNA',
)
sparse_writer.dump()
ds_sparse = scarf.DataStore(str(sparse_zarr))
ds_sparse

Minimum cell count (9) is lower than size factor multiplier (1000)



No matches found for pattern MT-|mt. Will not add/update percentage feature



No matches found for pattern RPS|RPL|MRPS|MRPL. Will not add/update percentage feature



More than of half of the less have less than 10 features for assay: RNA. Will not remove low quality cells automatically.



DataStore has 3 (3) cells with 1 assays: RNA
   Cell metadata:
            'I', 'ids', 'names', 'RNA_nFeatures', 'RNA_nCounts', 
          
   RNA assay has 0 (10) features and following metadata:
            'I', 'ids', 'names', 'nCells', 'dropOuts', 
          

In [15]:
for name, values in [
    ('toy_merge_a.zarr', [1, 2, 3, 4, 5, 6]),
    ('toy_merge_b.zarr', [7, 8, 9, 10, 11, 12]),
]:
    path = Path('scarf_datasets') / name
    if path.exists():
        shutil.rmtree(path)
    m = csr_matrix(np.asarray(values, dtype=np.uint16).reshape(3, 2))
    scarf.SparseToZarr(
        m,
        zarr_loc=str(path),
        cell_ids=[f'{path.stem}_{i}' for i in range(3)],
        feature_ids=['g1', 'g2'],
        assay_name='RNA',
    ).dump()

ds_a = scarf.DataStore('scarf_datasets/toy_merge_a.zarr')
ds_b = scarf.DataStore('scarf_datasets/toy_merge_b.zarr')
merger = scarf.DatasetMerge(
    datasets=[ds_a, ds_b],
    zarr_path='scarf_datasets/toy_merged.zarr',
    names=['a', 'b'],
    source_column='sample_id',
    overwrite=True,
)
merger.dump()
ds_merged = scarf.DataStore('scarf_datasets/toy_merged.zarr')
ds_merged.cells.head()

Minimum cell count (3) is lower than size factor multiplier (1000)



No matches found for pattern MT-|mt. Will not add/update percentage feature



No matches found for pattern RPS|RPL|MRPS|MRPL. Will not add/update percentage feature



More than of half of the less have less than 10 features for assay: RNA. Will not remove low quality cells automatically.



Minimum cell count (15) is lower than size factor multiplier (1000)



...

Minimum cell count (3) is lower than size factor multiplier (1000)



No matches found for pattern MT-|mt. Will not add/update percentage feature



No matches found for pattern RPS|RPL|MRPS|MRPL. Will not add/update percentage feature



More than of half of the less have less than 10 features for assay: RNA. Will not remove low quality cells automatically.



,I,ids,names,orig_RNA_nCounts,RNA_nFeatures,orig_RNA_nFeatures,RNA_nCounts,sample_id
0,True,a__toy_merge_a_2,toy_merge_a_2,11.0,2.0,2.0,11.0,a
1,True,a__toy_merge_a_1,toy_merge_a_1,7.0,2.0,2.0,7.0,a
2,True,a__toy_merge_a_0,toy_merge_a_0,3.0,2.0,2.0,3.0,a
3,True,b__toy_merge_b_2,toy_merge_b_2,23.0,2.0,2.0,23.0,b
4,True,b__toy_merge_b_1,toy_merge_b_1,19.0,2.0,2.0,19.0,b
